In [1]:
import pandas as pd
import numpy as np
import scipy
import sklearn

print("Môi trường đã sẵn sàng!")

Môi trường đã sẵn sàng!


Kiểm tra đường dẫn

In [2]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd()

INPUT_FILE = PROJECT_DIR / "data" / "All_Beauty_Cleaned_Merged.xlsx"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_FILE = PROCESSED_DIR / "interactions.parquet"

print("Thư mục dự án:", PROJECT_DIR)
print("File dữ liệu:", INPUT_FILE)

if not INPUT_FILE.is_file():
    raise FileNotFoundError(
        "Chưa tìm thấy dataset. Kiểm tra tên file và thư mục data."
    )

print("Đã tìm thấy dataset!")

Thư mục dự án: d:\Tài Liệu\RecommenderSystem
File dữ liệu: d:\Tài Liệu\RecommenderSystem\data\All_Beauty_Cleaned_Merged.xlsx
Đã tìm thấy dataset!


Đọc dữ liệu và lưu bản Parquet

In [3]:
COLUMNS = ["user_id", "parent_asin", "rating", "timestamp"]

cache_is_current = (
    PARQUET_FILE.exists()
    and PARQUET_FILE.stat().st_mtime >= INPUT_FILE.stat().st_mtime
)

if cache_is_current:
    print("Đang đọc bản Parquet...")
    df = pd.read_parquet(PARQUET_FILE, columns=COLUMNS)

else:
    print("Đang đọc Excel. File lớn nên bước này có thể mất vài phút...")

    df = pd.read_excel(
        INPUT_FILE,
        sheet_name="Merged_Data",
        usecols=COLUMNS,
        dtype={
            "user_id": "string",
            "parent_asin": "string",
            "rating": "float32",
            "timestamp": "int64",
        },
        engine="openpyxl",
    )

    df = df[COLUMNS]

    df.to_parquet(
        PARQUET_FILE,
        engine="pyarrow",
        index=False,
    )

    print("Đã lưu:", PARQUET_FILE)

print("Số dòng và số cột:", df.shape)
display(df.head())

Đang đọc Excel. File lớn nên bước này có thể mất vài phút...
Đã lưu: d:\Tài Liệu\RecommenderSystem\data\processed\interactions.parquet
Số dòng và số cột: (694252, 4)


,user_id,parent_asin,rating,timestamp
0,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,5.0,1588687728923
1,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,4.0,1588615855070
2,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,5.0,1589665266052
3,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B09JS339BZ,1.0,1643393630220
4,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B08BZ63GMJ,5.0,1609322563534


Kiểm tra dữ liệu trước khi chia tập

In [4]:
print(f"Tổng lượt đánh giá: {len(df):,}")
print(f"Số người dùng: {df['user_id'].nunique():,}")
print(f"Số sản phẩm: {df['parent_asin'].nunique():,}")

print("\nSố giá trị thiếu:")
display(df.isna().sum().to_frame("Số lượng thiếu"))

invalid_rating = ~df["rating"].isin([1, 2, 3, 4, 5])
print("Số rating không hợp lệ:", invalid_rating.sum())

duplicate_events = df.duplicated(
    subset=["user_id", "parent_asin", "timestamp"]
).sum()
print("Số tương tác trùng người dùng, sản phẩm và thời điểm:",
      duplicate_events)

user_activity = df.groupby("user_id").size()
print(f"Tỷ lệ người dùng chỉ có 1 đánh giá: "
      f"{user_activity.eq(1).mean():.2%}")

review_time = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
print("Thời gian đầu:", review_time.min())
print("Thời gian cuối:", review_time.max())

Tổng lượt đánh giá: 694,252
Số người dùng: 631,986
Số sản phẩm: 112,565

Số giá trị thiếu:


,Số lượng thiếu
user_id,0
parent_asin,0
rating,0
timestamp,0


Số rating không hợp lệ: 0
Số tương tác trùng người dùng, sản phẩm và thời điểm: 0
Tỷ lệ người dùng chỉ có 1 đánh giá: 93.19%
Thời gian đầu: 2000-11-01 04:24:18+00:00
Thời gian cuối: 2023-09-09 00:39:36.666000+00:00


Chia dữ liệu theo thời gian và lưu lại

In [5]:
import json

POSITIVE_THRESHOLD = 4.0

data = df.sort_values(
    "timestamp", kind="stable"
).reset_index(drop=True)

train_end = int(
    data["timestamp"].quantile(0.70, interpolation="nearest")
)
validation_end = int(
    data["timestamp"].quantile(0.90, interpolation="nearest")
)

train = data[data["timestamp"] <= train_end].copy()

val = data[
    (data["timestamp"] > train_end)
    & (data["timestamp"] <= validation_end)
].copy()

test = data[data["timestamp"] > validation_end].copy()

# Kiểm tra thứ tự thời gian và số lượng bản ghi
assert len(train) + len(val) + len(test) == len(data)
assert train["timestamp"].max() < val["timestamp"].min()
assert val["timestamp"].max() < test["timestamp"].min()

parts = {"train": train, "validation": val, "test": test}

for name, part in parts.items():
    part.to_parquet(
        PROCESSED_DIR / f"{name}.parquet",
        engine="pyarrow",
        index=False,
    )

config = {
    "split_method": "global_chronological",
    "train_end_timestamp_ms": train_end,
    "validation_end_timestamp_ms": validation_end,
    "positive_rating_threshold": POSITIVE_THRESHOLD,
    "item_column": "parent_asin",
}

(PROCESSED_DIR / "split_config.json").write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)

display(pd.DataFrame([
    {
        "Tập": name,
        "Số dòng": len(part),
        "Tỷ lệ": f"{len(part) / len(data):.2%}",
    }
    for name, part in parts.items()
]))

,Tập,Số dòng,Tỷ lệ
0,train,485977,70.00%
1,validation,138850,20.00%
2,test,69425,10.00%


Xây dựng Most Popular

In [6]:
from collections import defaultdict

REPORT_DIR = PROJECT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

train_positive = (
    train[train["rating"] >= POSITIVE_THRESHOLD]
    .drop_duplicates(["user_id", "parent_asin"])
)

# Danh sách sản phẩm đã xuất hiện trong train
catalog = pd.Index(
    train["parent_asin"].unique(),
    name="parent_asin",
)

popularity = (
    train_positive.groupby("parent_asin")
    .size()
    .reindex(catalog, fill_value=0)
    .rename("positive_user_count")
    .reset_index()
    .sort_values(
        ["positive_user_count", "parent_asin"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

popular_ids = popularity["parent_asin"].tolist()

# Lưu tất cả sản phẩm người dùng đã đánh giá trong train
seen_by_user = defaultdict(set)

for user_id, item_id in train[
    ["user_id", "parent_asin"]
].itertuples(index=False, name=None):
    seen_by_user[user_id].add(item_id)


def recommend_popular(user_id, k=10):
    if k < 1:
        raise ValueError("k phải lớn hơn hoặc bằng 1.")

    seen = seen_by_user.get(user_id, set())
    result = []

    for item_id in popular_ids:
        if item_id not in seen:
            result.append(item_id)

            if len(result) == k:
                break

    return result


popularity.to_csv(
    REPORT_DIR / "product_popularity.csv",
    index=False,
    encoding="utf-8-sig",
)

display(popularity.head(10))

,parent_asin,positive_user_count
0,B085BB7B1M,1552
1,B0BM4GX6TT,1127
2,B00R1TAN7I,973
3,B07C533XCW,922
4,B0107QYW14,765
5,B005BZQHEC,720
6,B09X9BG4FC,708
7,B0C9CWKY9G,699
8,B01M1OFZOG,688
9,B019GBG0IE,667


Tạo danh sách gợi ý cho một người dùng

In [7]:
demo_user = train_positive["user_id"].value_counts().index[0]

recommended_ids = recommend_popular(demo_user, k=10)

demo_result = (
    popularity.set_index("parent_asin")
    .loc[recommended_ids]
    .reset_index()
)

demo_result.insert(0, "rank", range(1, len(demo_result) + 1))

assert set(recommended_ids).isdisjoint(seen_by_user[demo_user])

print("Người dùng:", demo_user)
display(demo_result)

demo_result.to_csv(
    REPORT_DIR / "most_popular_demo.csv",
    index=False,
    encoding="utf-8-sig",
)

Người dùng: AG73BVBKUOH22USSFJA5ZWL7AKXA


,rank,parent_asin,positive_user_count
0,1,B085BB7B1M,1552
1,2,B0BM4GX6TT,1127
2,3,B00R1TAN7I,973
3,4,B07C533XCW,922
4,5,B0107QYW14,765
5,6,B005BZQHEC,720
6,7,B09X9BG4FC,708
7,8,B0C9CWKY9G,699
8,9,B01M1OFZOG,688
9,10,B019GBG0IE,667


Đánh giá Most Popular trên validation

In [8]:
import numpy as np

K = 10

val_positive = (
    val[val["rating"] >= POSITIVE_THRESHOLD]
    [["user_id", "parent_asin"]]
    .drop_duplicates()
)

truth_by_user = defaultdict(set)

for user_id, item_id in val_positive.itertuples(
    index=False, name=None
):
    if item_id not in seen_by_user.get(user_id, set()):
        truth_by_user[user_id].add(item_id)

warm_users = set(train_positive["user_id"])

# Trọng số giảm dần theo vị trí gợi ý
discounts = 1 / np.log2(np.arange(2, K + 2))

rows = []

for user_id, relevant_items in truth_by_user.items():
    recommendations = recommend_popular(user_id, k=K)

    hits = np.array(
        [item in relevant_items for item in recommendations],
        dtype=float,
    )

    hit_count = hits.sum()

    dcg = (hits * discounts[:len(hits)]).sum()
    ideal_dcg = discounts[:min(K, len(relevant_items))].sum()

    rows.append({
        "user_id": user_id,
        "has_positive_history": user_id in warm_users,
        "Precision@10": hit_count / K,
        "Recall@10": hit_count / len(relevant_items),
        "NDCG@10": dcg / ideal_dcg,
    })

per_user_metrics = pd.DataFrame(rows)

metric_columns = ["Precision@10", "Recall@10", "NDCG@10"]

groups = {
    "Toàn bộ": per_user_metrics,
    "Có lịch sử tích cực": per_user_metrics[
        per_user_metrics["has_positive_history"]
    ],
}

results = pd.DataFrame([
    {
        "Mô hình": "Most Popular",
        "Nhóm": name,
        "Số người dùng": len(group),
        **group[metric_columns].mean().to_dict(),
    }
    for name, group in groups.items()
])

display(results)

results.to_csv(
    REPORT_DIR / "most_popular_validation.csv",
    index=False,
    encoding="utf-8-sig",
)

,Mô hình,Nhóm,Số người dùng,Precision@10,Recall@10,NDCG@10
0,Most Popular,Toàn bộ,85951,0.001437,0.014173,0.006655
1,Most Popular,Có lịch sử tích cực,4392,0.000888,0.008500,0.004118
